In [2]:
import jax
jax.config.update("jax_enable_x64", True)
from jax.lax import integer_pow
from gwfast.gwfastGlobals import DAY_TO_SEC
from gwfast.lensing_utils import compute_lensed_angles_approx

from functools import partial
from multiprocessing import Pool, cpu_count
from typing import Union
from collections import OrderedDict

from pathlib import Path
from jax import config, tree, vmap, random, jacfwd
import jax.numpy as np
import numpy as onp
from jax.lax import broadcast_in_dim
config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt

from gwfast.gwfastGlobals import detectors as det_dict, detPath, MTSUN_SI, DAY_TO_SEC
import gwfast.waveforms as waveforms
from gwfast.detector import Detector
import gwfast.network as network
from gwfast.signals import AGNLensedGWSignal, GeneralLensedGWSignal
from gwfast.fisherTools import (
    reduce_Fisher_matrix, 
    plot_corners, compute_covariance_matrix, 
    print_matrices, covariance_change_variable
)


/users/hin-wai.leong/src/AGN-gwfast/gwfast/waveforms.py:35: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


In [27]:
def _high_dim_matmul_2(mat_1, mat_2):
    """
    Batched matrix multiplication over trailing (event/grid) axes.

    Expected shapes:
      mat_1: (N, M, *S)
      mat_2: (M, K, *S)  or (M, K) (broadcast across *S)

    Returns:
      out: (N, K, *S)
    """
    if mat_1.ndim < 2 or mat_2.ndim < 2:
        raise ValueError(f"high_dim_matmul expects arrays with ndim>=2, got {mat_1.ndim}, {mat_2.ndim}")

    N, M = mat_1.shape[0], mat_1.shape[1]
    if mat_2.shape[0] != M:
        raise ValueError(f"inner dim mismatch: mat_1 is ({N},{M},...), mat_2 is ({mat_2.shape[0]},{mat_2.shape[1]},...)")

    # Simple 2D case
    if mat_1.ndim == 2 and mat_2.ndim == 2:
        return mat_1 @ mat_2

    S1 = mat_1.shape[2:]  # batch shape
    K = mat_2.shape[1]

    # Allow mat_2 to be broadcast (constant across batch)
    if mat_2.ndim == 2:
        # reshape to (1, M, K) then broadcast later
        mat_2_batched = mat_2[None, :, :]
        S2 = S1
    else:
        S2 = mat_2.shape[2:]
        if S2 != S1:
            raise ValueError(f"batch shape mismatch: mat_1 batch={S1}, mat_2 batch={S2}")
        # Move batch axes to the front, then flatten
        perm2 = (*range(2, mat_2.ndim), 0, 1)  # (*S, M, K)
        mat_2_batched = np.transpose(mat_2, axes=perm2).reshape((-1, M, K))  # (B, M, K)

    # Move batch axes to the front, then flatten
    perm1 = (*range(2, mat_1.ndim), 0, 1)  # (*S, N, M)
    mat_1_batched = np.transpose(mat_1, axes=perm1).reshape((-1, N, M))  # (B, N, M)

    # If mat_2 is constant across the batch, broadcast it to (B, M, K)
    if mat_2.ndim == 2:
        B = mat_1_batched.shape[0]
        mat_2_batched = np.broadcast_to(mat_2_batched, (B, M, K))

    out_batched = mat_1_batched @ mat_2_batched  # (B, N, K)

    # Unflatten batch and move axes back to (N, K, *S)
    out = out_batched.reshape((*S1, N, K))
    out = np.transpose(out, axes=(len(S1), len(S1) + 1, *range(0, len(S1))))
    return out


def _high_dim_matmul_1(mat_1, mat_2):
    """
    Perform matrix multiplication for high-dimensional arrays.
    The first two dimensions of the input arrays are treated as matrices.
    """
    return np.einsum('ij...,jk...->ik...', mat_1, mat_2)


def high_dim_matmul(mat_1, mat_2, style='1'):
    if style == '1':
        return _high_dim_matmul_1(mat_1, mat_2)
    elif style == '2':
        return _high_dim_matmul_2(mat_1, mat_2)
    else:
        raise ValueError(f"Invalid style '{style}' for high_dim_matmul. Choose '1' or '2'.")


def covariance_change_variable(
        convariance_matrix, injection_parameters, transform, from_params
    ):

    np = onp
    np.set_printoptions(precision=3)
    full_rank = convariance_matrix.shape[0]
    param_shape = convariance_matrix.shape[2:]  # empty () if convariance_matrix is 2D
    matrix_keys = list(injection_parameters.keys())
    keys_indices = [matrix_keys.index(key) for key in from_params]
    # print(keys_indices)

    # Flatten inputs (support scalars too)
    sub_injection_parameters = OrderedDict(
        {key: np.atleast_1d(injection_parameters[key]).reshape(-1) for key in from_params}
    )

    sub_transformed_parameters = transform(sub_injection_parameters)
    out_keys = list(sub_transformed_parameters.keys())

    # Decide whether we have a batch axis to vmap over
    # If covariance is 2D, we treat it as a single event and DO NOT vmap.
    if convariance_matrix.ndim == 2:
        jacobian_pytree = jacfwd(transform)(OrderedDict(
            {k: sub_injection_parameters[k][0] for k in from_params}
        ))
        # jacobian_pytree[out][in] are scalars
        jacobian_mat = np.stack(
            [
                np.stack([np.asarray(jacobian_pytree[ok][ik]) for ik in from_params], axis=0)
                for ok in out_keys
            ],
            axis=0
        )  # (Nout, Nin)

        full_jacobian_mat = np.eye(full_rank, dtype=convariance_matrix.dtype)
        full_jacobian_mat[np.ix_(keys_indices, keys_indices)] = jacobian_mat

        J = full_jacobian_mat
        transform_covar = J @ convariance_matrix @ J.T

        print(onp.linalg.cond(full_jacobian_mat.astype(onp.float64)))
        print(
        onp.diag(full_jacobian_mat), \
            onp.max(onp.abs(full_jacobian_mat)), \
                onp.min(onp.abs(full_jacobian_mat))
                )

        sign, logdet = np.linalg.slogdet(convariance_matrix.astype(np.float64))
        print('Covariance', sign, logdet)
        sign, logdet = np.linalg.slogdet(J.astype(np.float64))
        print('Jacobian', sign, logdet)
        sign, logdet = np.linalg.slogdet(J.T.astype(np.float64))
        print('Jacobian T', sign, logdet)

    else:
        # Batched case: vmap over event axis
        # Ensure leaves are rank>=1 for vmap
        _scalar_keys = [k for k, v in sub_injection_parameters.items() if np.ndim(v) == 0]
        if _scalar_keys:
            raise ValueError(f"Expected batched inputs, got scalars for keys={_scalar_keys}")

        jacobian_pytree = vmap(jacfwd(transform))(sub_injection_parameters)
        # dict[out][in] -> (Nflat,)
        jacobian_mat = np.stack(
            [
                np.stack([np.asarray(jacobian_pytree[ok][ik]) for ik in from_params], axis=0)
                for ok in out_keys
            ],
            axis=0
        ).reshape(len(out_keys), len(from_params), *param_shape)

        full_jacobian_mat = np.zeros_like(convariance_matrix)
        full_jacobian_mat[np.diag_indices(full_rank)] = 1.0
        full_jacobian_mat[np.ix_(keys_indices, keys_indices)] = jacobian_mat
        full_jacobian_mat_moved = np.moveaxis(full_jacobian_mat, -1, 0)
        # print(full_jacobian_mat_moved)

        full_jacobian_mat_T = np.transpose(
            full_jacobian_mat, axes=(1, 0, *range(2, full_jacobian_mat.ndim))
        )
        for idx in range(full_jacobian_mat.shape[2]):
            mat = full_jacobian_mat[:, :, idx]
            jac_mat = mpmath.matrix(mat)
            try:
                eigv = np.array(mpmath.eigh(jac_mat)[0], dtype=np.float128)
                abs_eigv = np.abs(eigv)
                print('Transformation')
                _det = np.prod(eigv)
                print(np.array([np.sign(_det), np.log(np.abs(_det)), np.max(abs_eigv) / np.min(abs_eigv)]))
                print(eigv)
                print(np.diag(mat))
                # print(mat)
                det = np.prod(eigv)
                print(np.sign(det), np.log(np.abs(det)))
            except RuntimeError:
                print('Jacobian is not diagonalizable')

            sign, _ = np.linalg.slogdet(mat)
            print(idx, sign)

            cov_mat = mpmath.matrix(convariance_matrix[:, :, idx])

            print('MPMath Trans-Cov')
            trans_cov = jac_mat * cov_mat * jac_mat.T
            print('Original Cov.')
            try:
                eigv = np.array(mpmath.eigh(cov_mat)[0], dtype=np.float128)
                _det = np.prod(eigv)
                print(np.array([np.sign(_det), np.log(np.abs(_det)), np.max(np.abs(eigv)) / np.min(np.abs(eigv))]))
                print(eigv)
                det = np.prod(eigv)
                print(np.sign(det), np.log(np.abs(det)))
            except:
                print('Covariance is not diagonalizable')
            print('Transformed Cov')
            try:
                eigv = np.array(mpmath.eigh(trans_cov)[0], dtype=np.float128)
                abs_eigv = np.abs(eigv)
                _det = np.prod(eigv)
                print(np.array([np.sign(_det), np.log(np.abs(_det)), np.max(abs_eigv) / np.min(abs_eigv)]))
                print(eigv)
                det = np.prod(eigv)
                print(np.sign(det), np.log(np.abs(det)))
            except RuntimeError:
                print('Transformed Cov is not diagonalizable')
            print('-------------------')


        transform_covar = high_dim_matmul(
            full_jacobian_mat, high_dim_matmul(convariance_matrix, full_jacobian_mat_T)
        )

    # Maintain original order of keys, replacing transformed subset
    transform_keys = list(matrix_keys)
    for idx, new_key_name in zip(keys_indices, out_keys):
        transform_keys[idx] = new_key_name

    transform_parameters = {}
    for key in transform_keys:
        value = injection_parameters.get(key, None)
        if value is None:
            value = sub_transformed_parameters.get(key, None)
        # For 2D covariance, param_shape=() so reshape is a no-op for scalars/arrays
        transform_parameters[key] = np.asarray(value).reshape(*param_shape) if convariance_matrix.ndim > 2 else value

    return transform_covar, transform_parameters, transform_keys


def lensing_transform(lensing_parameters):
    # A fiducial phase which does not affect the Jacobian results
    lensing_parameters['phase'] = np.zeros_like(lensing_parameters['iota']) + 0.1
    lensing_parameters['psi'] = np.zeros_like(lensing_parameters['iota']) + 0.1
    outputs = compute_lensed_angles_approx(lensing_parameters)
    phenom_changes = {}
    phenom_changes['delta_iota'] = outputs['iota_m'] - outputs['iota_p']
    phenom_changes['delta_phi'] = outputs['phase_m'] - outputs['phase_p']

    # (Radial gravitational potential is cancelled)
    phenom_changes['relative_mass'] = (1 + outputs['z_rel_m']) / (1 + outputs['z_rel_p'])
    relative_magification = outputs['sqrt_mu_p'] / outputs['sqrt_mu_m']
    phenom_changes['relative_distance'] = \
        relative_magification * integer_pow((1 + outputs['z_rel_m']) / (1 + outputs['z_rel_p']), 2)
    phenom_changes['delta_time'] = outputs['delta_time'] # * DAY_TO_SEC
    return phenom_changes

In [46]:
lensing_transform(lensing_parameters)

{'delta_iota': Array([0.14804162, 0.0616347 , 0.02539659, 0.00979653, 0.00063313],      dtype=float64),
 'delta_phi': Array([-0.2410353 , -0.27607964, -0.28173498, -0.28270788, -0.2828769 ],      dtype=float64),
 'relative_mass': Array([0.98310026, 0.98076802, 0.98075538, 0.98242784, 0.99736085],      dtype=float64),
 'relative_distance': Array([0.78202223, 0.58124159, 0.3011035 , 0.08557613, 0.00284709],      dtype=float64),
 'delta_time': Array([   3.07245264,    7.43681362,   18.80510489,   57.94187044,
        1304.75433415], dtype=float64)}

In [162]:
# a random covariance matrix
seed = 42
key = random.key(seed)

dim = 14
matrices = onp.random.uniform(size=(dim, dim, 2)).astype(onp.float128)
sym_mat = 0.5 * (matrices + matrices.transpose((1, 0, 2)) ) + dim * onp.eye(dim)[:, :, None]  # add small diagonal for positive-definiteness



In [73]:
sign, logdet = np.linalg.slogdet(sym_mat[:, :, 1])
sign

TypeError: Dtype float128 is not a valid JAX array type. Only arrays of numeric types are supported by JAX.

In [330]:
transformed_keys

['Mc',
 'eta',
 'delta_iota',
 'phase',
 'chi1z',
 'chi2z',
 'tcoal',
 'delta_phi',
 'relative_distance',
 'relative_mass',
 'delta_time',
 'psi',
 'theta',
 'phi']

In [324]:
DAY_TO_SEC

86400.0

In [41]:
# a random covariance matrix
seed = 42
key = random.key(seed)

dim = 11
matrices = onp.random.uniform(size=(dim, dim, 2)).astype(onp.float128) * 5
sym_mat = 0.5 * (matrices + matrices.transpose((1, 0, 2)) ) + 10 * dim * onp.eye(dim)[:, :, None]  # add small diagonal for positive-definiteness



reference_parameters = {
    'Mc': 30, 'eta': 0.24, 'iota': 0.9*np.pi/2, 'phase': 2,
    'chi1z': 0.3, 'chi2z': 0.5, 'tcoal': 0,
    'R_orbit': 100, 'M_lz': 1e4, 'src_pos': 0.5,
    'dL': 1, 'psi': 4, 'theta': 1.87, 'phi': 2.66,
}

from_params = ['iota', 'R_orbit', 'src_pos', 'M_lz', 'dL']

lensing_parameters = {key: np.full(2, val).astype(np.float64) for key, val in reference_parameters.items()}

transformed_cov_mat, transformed_parameters, transformed_keys = covariance_change_variable(
        sym_mat[:, :, 1], lensing_parameters, lensing_transform, from_params
    )

dt_idx = transformed_keys.index('delta_time')
transformed_cov_mat[dt_idx, :] *= DAY_TO_SEC
transformed_cov_mat[:, dt_idx] *= DAY_TO_SEC

# print(np.linalg.slogdet(transformed_cov_mat[:, :], method='lu'))
# print(np.linalg.slogdet(transformed_cov_mat[:, :], method='qr'))

print(onp.linalg.cond(sym_mat[:, :, 1].astype(onp.float64)))
print(onp.linalg.cond(transformed_cov_mat[:, :].astype(onp.float64)))

trans_cov_mat_64 = transformed_cov_mat.astype(onp.float64)
print(onp.linalg.slogdet(trans_cov_mat_64))
logdeg = onp.linalg.det(trans_cov_mat_64)
print(np.sign(logdeg), onp.log(np.abs(logdeg)))

# L = onp.linalg.cholesky(trans_cov_mat_64)
# print(onp.sum(onp.log(onp.diag(L))) * 2)


[2, 7, 9, 8, 10]
5.1696400249499845e+19
[ 1.00000000e+00  1.00000000e+00 -5.02187348e-01  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00 -1.37691022e-03
  4.56703816e-19 -6.46579595e-03  2.67043547e-01] 69.7420342588674913 0.0
Covariance 1.0 51.920278397778894
Jacobian 1.0 -52.25122770960254
Jacobian T 1.0 -52.25122770960254
1.3375953627885624
5.581427262408606e+29
(-1.0, 23.0179076639909)
-1.0 23.0179076639909


In [197]:
onp.diag(transformed_cov_mat), \
      onp.max(onp.abs(transformed_cov_mat)), \
        onp.min(onp.abs(transformed_cov_mat))

(array([1.10518674e+02, 1.13601027e+02, 3.35824167e+01, 1.11495321e+02,
        1.14872725e+02, 1.13392811e+02, 1.12071857e+02, 1.58768717e+00,
        5.42399509e+05, 2.07598893e-02, 4.36775681e+03], dtype=float128),
 542399.50925348438204,
 0.00080998869525121038854)

-28.481741560420527


In [66]:
np.log10(np.max(transformed_cov_mat[np.diag_indices(dim)]) / np.min(transformed_cov_mat[np.diag_indices(dim)]))

Array(7.77173913, dtype=float64)

In [49]:
print(np.linalg.det(transformed_cov_mat[:, :]))
print(onp.linalg.det(transformed_cov_mat[:, :]))

3.092342891288947e-11
3.092342891288969e-11


In [33]:
np.linalg.slogdet?

Signature:      np.linalg.slogdet(a: 'ArrayLike', *, method: 'str | None' = None) -> 'SlogdetResult'
Call signature: np.linalg.slogdet(*args, **kwargs)
Type:           PjitFunction
String form:    <PjitFunction of <function slogdet at 0x7f324ee11c60>>
File:           ~/.conda/envs/180125_py311_AGNLensing/lib/python3.11/site-packages/jax/_src/numpy/linalg.py
Docstring:     
Compute the sign and (natural) logarithm of the determinant of an array.

JAX implementation of :func:`numpy.linalg.slotdet`.

Args:
  a: array of shape ``(..., M, M)`` for which to compute the sign and log determinant.
  method: the method to use for determinant computation. Options are

    - ``'lu'`` (default): use the LU decomposition.
    - ``'qr'``: use the QR decomposition.

Returns:
  A tuple of arrays ``(sign, logabsdet)``, each of shape ``a.shape[:-2]``

  - ``sign`` is the sign of the determinant.
  - ``logabsdet`` is the natural log of the determinant's absolute value.

See also:
  :func:`jax.numpy.linalg

In [4]:
reference_parameters = {
    'Mc': 30, 'eta': 0.24, 'iota': 0.99*np.pi/2, 'phase': 2,
    'chi1z': 0.3, 'chi2z': 0.5, 'tcoal': 0,
    'R_orbit': 100, 'M_lz': 1e4, 'src_pos': 0.5,
    'dL': 1, 'psi': 4, 'theta': 1.87, 'phi': 2.66,
}

len(reference_parameters.keys())

14

In [5]:
# Set up detectors
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

wf_model = waveforms.IMRPhenomD()

H1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=H1, fmin=10)
L1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=L1, fmin=10)
V1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=V1, fmin=10)
HLV_AGN = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1


In [33]:
shape = 8
lensing_parameters = {
    'Mc': 80, 'eta': 0.23, 'iota': 0.99 * np.pi / 2, 'phase': 2.1,
    'chi1z': 0.3, 'chi2z': 0.4, 'tcoal': 0.0,
    'theta': 0.1, 'phi': 0.3, 'psi': 0.3, 'dL': 2.0, 
    'R_orbit': 100, 'M_lz': 5e5, 'src_pos': 0.2
}
lensing_parameters = {key: np.full(shape, val).astype(np.float64) \
                      for key, val in lensing_parameters.items()}
lensing_parameters['src_pos'] = - np.hstack([np.geomspace(0.003, 0.06, 5), np.geomspace(0.07, 0.96, 3)])


HLV_fisher = HLV_AGN.FisherMatr(lensing_parameters, res=1000)

Computing Fisher for H1...


Computing Fisher for L1...
Computing Fisher for V1...
Done.


In [31]:
np.geomspace([0.03, 0.07, 5])

TypeError: geomspace() missing 1 required positional argument: 'stop'

In [32]:
np.hstack([np.geomspace(0.003, 0.06, 5), np.geomspace(0.07, 0.99, 3)])

Array([0.003, 0.006, 0.013, 0.028, 0.06 , 0.07 , 0.263, 0.99 ], dtype=float64)

In [39]:
transformed_keys

['Mc',
 'eta',
 'delta_iota',
 'phase',
 'chi1z',
 'chi2z',
 'tcoal',
 'theta',
 'phi',
 'psi',
 'delta_time',
 'relative_mass',
 'relative_distance',
 'delta_phi']

In [30]:
from_params = ['iota', 'src_pos', 'R_orbit', 'M_lz', 'dL']

cov_mats, _ = compute_covariance_matrix(HLV_fisher[:, :], cores=4)

transformed_cov_mat, transformed_parameters, transformed_keys = \
    covariance_change_variable(
        cov_mats.astype(np.float64), lensing_parameters, lensing_transform, from_params
    )

# print(onp.linalg.cond(sym_mat[:, :, 1].astype(onp.float64)))
for idx in range(0, transformed_cov_mat.shape[2]):
    mat = transformed_cov_mat[:, :, idx]
    mat_f8 = mat.astype(onp.float64)
    print('---------')
    print(idx, onp.linalg.cond(mat_f8))

    print(idx, onp.linalg.slogdet(mat_f8))
    # logdeg = onp.linalg.det(trans_cov_mat_64)
    # print(np.sign(logdeg), onp.log(np.abs(logdeg)))

# print(np.linalg.slogdet(transformed_cov_mat[:, :], method='lu'))
# print(np.linalg.slogdet(transformed_cov_mat[:, :], method='qr'))

# idx = 0
# mat = transformed_cov_mat
# mat_f8 = mat.astype(onp.float64)
# print('---------')
# print(idx, onp.linalg.cond(mat_f8))
# print(idx, onp.linalg.slogdet(mat_f8))

# L = onp.linalg.cholesky(trans_cov_mat_64)
# print(onp.sum(onp.log(onp.diag(L))) * 2)


/users/hin-wai.leong/.conda/envs/180125_py311_AGNLensing/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Jacobian is not diagonalizable
0 1.0
MPMath Trans-Cov
Original Cov.
Covariance is not diagonalizable
Transformed Cov


/users/hin-wai.leong/.conda/envs/180125_py311_AGNLensing/lib/python3.11/site-packages/numpy/linalg/linalg.py:2079: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


Transformed Cov is not diagonalizable
-------------------
Jacobian is not diagonalizable
1 1.0
MPMath Trans-Cov
Original Cov.
Covariance is not diagonalizable
Transformed Cov
Transformed Cov is not diagonalizable
-------------------
Jacobian is not diagonalizable
2 1.0
MPMath Trans-Cov
Original Cov.
Covariance is not diagonalizable
Transformed Cov
Transformed Cov is not diagonalizable
-------------------
Transformation
[-1.000e+00 -2.348e+00  1.605e+05]
[-1.021e+02 -8.863e+00 -1.558e-03  6.496e-04  1.000e+00  1.000e+00
  1.000e+00  1.000e+00  1.000e+00  1.000e+00  1.000e+00  1.000e+00
  1.000e+00  1.042e+02]
[ 1.000e+00  1.000e+00 -8.878e+00  1.000e+00  1.000e+00  1.000e+00
  1.000e+00  1.000e+00  1.000e+00  1.000e+00 -3.719e-01  1.713e-04
  9.852e-22  2.483e+00]
-1.0 -2.3484681787683698573
3 -1.0
MPMath Trans-Cov
Original Cov.
[-1.000e+00 -7.463e+01  1.989e+18]
[-1.190e-06  2.776e-09  1.347e-06  1.189e-05  1.693e-05  2.728e-05
  1.476e-04  1.442e-03  6.300e-03  1.409e-02  2.249e+00  3

LinAlgError: SVD did not converge

## Well, let's try to use float128 for determinant computation as well.

In [10]:
import mpmath

idx = 0
mat = transformed_cov_mat[:, :, idx]
mp_fisher = mpmath.matrix(mat)
P, L, U = mpmath.lu(mp_fisher)

NameError: name 'transformed_cov_mat' is not defined

In [28]:
eigv = onp.array(mpmath.eigh(mp_fisher)[0], dtype=onp.float128)
min(eigv)

-4.2401392717272265998

In [34]:
idx = 3
mat = transformed_cov_mat[:, :, idx]
mp_fisher = mpmath.matrix(mat)
det = onp.prod(onp.array(mpmath.eigh(mp_fisher)[0], dtype=onp.float128))
print(onp.sign(det), onp.log(onp.abs(det)))

-1.0 92.952741266480478695


- [ ] Check the cov. matrix entries, MLz, y and R
- [ ] Maths: determinant relations
- [ ] Check it in the real-world example
- [ ] Check if the $y$ threshold is the issue
- [ ] Weird matrices -> relation does not hold?
- [ ] Check the orig. covariance determinant

Bounded from above: (possibly) the $y$ threshold
Bounded from below: the determinant of the orignal cov. is too small. (not sure why...)

`slogdet` > `det` 